# IDTrack Tutorials — What IDTrack Is, How to Think About It, and How to Use These Notebooks

*Last updated:* 2026-01-07

These tutorials are written for **wet-lab and general biology audiences** who need to *translate and harmonize gene identifiers*
across datasets, time, and databases. They assume **no software development background**, but they do go deep enough to help you
use IDTrack confidently in real analysis work.


## 0. What problem does IDTrack solve?

In modern biology you rarely analyze a single dataset in isolation. You compare:
- older vs newer studies (different Ensembl releases)
- different technologies (bulk vs scRNA-seq)
- different annotation sources (Ensembl IDs vs HGNC symbols vs RefSeq vs UniProt)

The challenge: **identifiers change**.
- Ensembl stable IDs can be *retired*, *merged*, *split*, or *re-assigned across releases*.
- Gene symbols are convenient but can be **ambiguous** and can change over time.
- External databases provide cross-references, but they overlap and can introduce many-to-many ambiguity.

IDTrack is designed to make these changes **explicit, reproducible, and auditable** instead of hidden and ad hoc.


## 1. The mental model (the one you should remember)

IDTrack is easiest to understand with two axes:

1. **Time axis** → *Ensembl releases* (e.g. 90, 100, 110, 115)
2. **Space axis** → *identifier namespaces* (Ensembl gene IDs, gene symbols, UniProt accessions, RefSeq IDs, …)

Most real tasks are:

- **Time travel on the backbone** (move an Ensembl identifier from one release to another)
- then optionally **switch namespaces** (e.g. Ensembl → HGNC Symbol, or Ensembl → UniProt)

A useful picture:

```text
Your ID (some database, some year)
        |
        v
  [Normalize + match to graph node]
        |
        v
  [Time travel across Ensembl releases]
        |
        +--> (optional) [external hop(s) if needed]
        |
        v
  [Arrive at target release]
        |
        v
  (optional) [Convert to requested external database]
```


## 2. What you build once (and reuse many times): the graph snapshot

IDTrack builds a **graph** where:
- nodes are identifiers (Ensembl IDs, base IDs, external IDs)
- edges encode relationships (release-to-release history, gene↔transcript↔protein links, external cross-references)

A key feature is the **snapshot release**:
- you choose a *maximum Ensembl release* (e.g. 115)
- IDTrack ignores anything newer

This matters because it makes your results **reproducible**:
- same snapshot release + same external configuration → same graph → same conversions


## 3. What you edit as a user: the external YAML

IDTrack does *not* automatically include every external database Ensembl knows about.
Instead, you explicitly opt in via a small YAML file.

Why? Because including everything would:
- make the graph huge
- slow down path-finding
- increase ambiguity (many-to-many relationships)

So the workflow is:
1. generate a template YAML for an organism
2. enable a curated set of external databases (set `Include: true`)
3. build the graph


## 4. What counts as a ‘good’ result? (1→0, 1→1, 1→n)

When you convert one identifier, three outcomes are common:

- **1→0**: nothing matches (unknown ID, or no path exists)
- **1→1**: clean conversion
- **1→n**: ambiguous conversion (splits, merged history, promiscuous external IDs, symbols, …)

IDTrack will *tell you which case you are in*. This is a feature, not a failure: it prevents silent mistakes.


## 5. Assumptions and practical requirements

IDTrack assumes you have:
- a writable **local repository folder** (cache directory)
- network access the first time you build a graph (downloads metadata and tables)
- enough disk space (graphs and cached tables can be large)

You control where things go via `IDTRACK_LOCAL_REPO`.


In [ ]:
# 1) Minimal setup cell (safe to run in any notebook)
from __future__ import annotations

import os
from pathlib import Path

import idtrack

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', './idtrack_cache')).resolve()
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api.configure_logger()

print('IDTrack version:', idtrack.__version__)
print('Local repository:', LOCAL_REPOSITORY)


The cell above does three things:
1. chooses a cache directory (your *local repository*)
2. creates the high-level `idtrack.API` object
3. enables logging so you can see progress (downloads, caching, graph build)


In [ ]:
# 2) Resolve organism names the way IDTrack expects
# You can use common names ('human'), scientific names, taxon IDs, or Ensembl-style names.

for query in ['human', 'mus musculus', 'sus scrofa']:
    formal_name, latest_release = api.resolve_organism(query)
    print(f'{query!r} -> {formal_name!r} (latest Ensembl release: {latest_release})')


You will see outputs like:
- `'human' -> 'homo_sapiens'`
- `'mus musculus' -> 'mus_musculus'`
- `'sus scrofa' -> 'sus_scrofa'`

From now on, the tutorials use the **formal Ensembl names** (snake_case).


## 6. The main user-facing building blocks

You do not need to be a developer to use these, but knowing the names helps you navigate the tutorials:

1. **`idtrack.API`** — the high-level entry point
   - resolves organism names
   - builds/loads the graph snapshot
   - provides `convert_identifier(...)` and batch helpers

2. **`idtrack.DatabaseManager`** — data access + caching
   - downloads tables from Ensembl MySQL
   - manages your external YAML (`*_externals_modified.yml`)

3. **`idtrack.Track`** — the conversion engine
   - performs path-finding and scoring in the graph
   - you usually access it as `api.track`

4. **`idtrack.HarmonizeFeatures`** — multi-dataset harmonization
   - converts gene identifiers in multiple `.h5ad` datasets
   - helps build a unified integrated dataset

5. **`idtrack._external_mappers` (optional)** — orthologs and external mapping services
   - advanced features that require extra dependencies


## 7. Tutorial roadmap (recommended order)

1. **Prepare external YAMLs** (human, mouse, pig)
2. **Build graph snapshots** for each organism
3. **Run self-tests / sanity checks** (confirm your setup is correct)
4. **Human API deep dive** (how to actually convert IDs and interpret results)
5. **HLCA / dataset harmonization tutorial** (practical single-cell application)


## 8. Quick troubleshooting (fast wins)

If something fails, check these first:

1. **Permissions**: is your local repository writable?
2. **Network**: first-time runs need to reach Ensembl services (REST + MySQL).
3. **Disk space**: building graphs can use multiple GB.
4. **Snapshot release**: if you pick a release not available for an organism/assembly, build fails.
5. **External YAML**: for mouse/pig you must create a `*_externals_modified.yml` in your local repository.
